Time to take our raw audio files and convert them into a feature matrix that can be used for genre classification.

In [2]:
import librosa
import numpy as np
import pandas as pd
import os
from pathlib import Path

First up, define the function that takes an audio chunk and generates features. Focusing broadly on overall means and standard deviations for the main features but will likely come back to do more rich feature engineering if needed. Want to start simple.

In [8]:
def extract_features(y, sr):
    features = {}
    
    # MFCCs — mean and std for each coefficient
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    for i in range(13):
        features[f'mfcc{i+1}_mean'] = mfccs[i].mean()
        features[f'mfcc{i+1}_std'] = mfccs[i].std()
    
    # Spectral centroid
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    features['spectral_centroid_mean'] = centroid.mean()
    features['spectral_centroid_std'] = centroid.std()
    
    # Spectral rolloff
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    features['spectral_rolloff_mean'] = rolloff.mean()
    features['spectral_rolloff_std'] = rolloff.std()
    
    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(y)
    features['zcr_mean'] = zcr.mean()
    features['zcr_std'] = zcr.std()
    
    # Chroma
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    features['chroma_mean'] = chroma.mean()
    features['chroma_std'] = chroma.std()
    
    # RMS energy
    rms = librosa.feature.rms(y=y)
    features['rms_mean'] = rms.mean()
    features['rms_std'] = rms.std()
    
    # Tempo
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    features['tempo'] = float(tempo.item())
    
    return features

Next, we'll take one of the full audio files (30 seconds for our training set) and split it into 3 second chunks. The idea here is to effectively "create" more data by getting more features out of individual songs. The one caveat is that we have to be careful during cross validation to make sure that there isn't leakage from training data into test data - if we include a chunk of a song in the training data then including a chunk of the same song in test means the model will have effectively already learned from that song. Separating our train / test split by the source fill later will prevent that issue.

In [4]:
def process_file(file_path, genre, chunk_duration=3, sr=22050):
    chunks = []
    
    try:
        # Load full audio file.
        y, sr = librosa.load(file_path, sr=sr, mono=True)
        
        # Calculate samples per chunk
        chunk_samples = chunk_duration * sr
        total_samples = len(y)

        # Floor division drops an incomplete final chunk.
        num_chunks = total_samples // chunk_samples
        
        for i in range(num_chunks):
            start = i * chunk_samples
            end = start + chunk_samples
            chunk = y[start:end]
            
            # Extract features for this chunk
            features = extract_features(chunk, sr)
            
            # Add metadata — used for train / test split later on.
            features['genre'] = genre
            features['source_file'] = Path(file_path).name
            features['chunk_id'] = i
            
            chunks.append(features)
            
    except Exception as e:
        print(f'Error processing {file_path}: {e}')
    
    return chunks

And now simply to define the loop over all our files. Repo is assumed to be located at `data/genres_original/genre/file.wav`

In [5]:
def build_dataset(data_dir, chunk_duration=3):
    all_chunks = []
    data_path = Path(data_dir)
    
    genres = [d.name for d in data_path.iterdir() if d.is_dir()]
    print(f'Found genres: {genres}')
    
    for genre in genres:
        genre_path = data_path / genre
        files = list(genre_path.glob('*.wav'))
        print(f'Processing {genre}: {len(files)} files...')
        
        for file_path in files:
            chunks = process_file(str(file_path), genre, chunk_duration)
            all_chunks.extend(chunks)
    
    df = pd.DataFrame(all_chunks)
    print(f'\nDone! Total chunks: {len(df)}')
    print(f'Features per chunk: {len(df.columns) - 3}')  # subtract genre, source_file, chunk_id
    return df

Let's run it.

In [9]:
df = build_dataset('data/genres_original')

df.to_csv('data/features.csv', index=False)
print(df.head())
print(df.shape)

Found genres: ['pop', 'metal', 'disco', 'blues', 'reggae', 'classical', 'rock', 'hiphop', 'country', 'jazz']
Processing pop: 100 files...
Processing metal: 100 files...
Processing disco: 100 files...
Processing blues: 100 files...
Processing reggae: 100 files...
Processing classical: 100 files...
Processing rock: 100 files...
Processing hiphop: 100 files...
Processing country: 100 files...
Processing jazz: 100 files...


/var/folders/0j/45q1_hkx01sd7_mfg_4ph6mh0000gn/T/ipykernel_24106/2397307509.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(file_path, sr=sr, mono=True)
/Users/jvonderwell/mamb/envs/6250Project/lib/python3.11/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Error processing data/genres_original/jazz/jazz.00054.wav: 

Done! Total chunks: 9981
Features per chunk: 37
   mfcc1_mean  mfcc1_std  mfcc2_mean  mfcc2_std  mfcc3_mean  mfcc3_std  \
0  -93.494385  66.824570   69.222076  29.108154   10.906341  14.842863   
1  -84.898590  57.393242   76.788597  31.295662    8.512515  26.066940   
2 -111.756813  72.142387   79.589035  29.334457   18.292074  19.606325   
3  -78.309921  61.646477   81.384483  31.859398   18.922922  16.392584   
4 -107.066513  69.686134   90.173180  29.383104   22.560926  13.423120   

   mfcc4_mean  mfcc4_std  mfcc5_mean  mfcc5_std  ...  zcr_mean   zcr_std  \
0   19.867382  17.622301   17.565256  11.632965  ...  0.116230  0.053967   
1    7.267838  14.072378   12.760130  16.203993  ...  0.109266  0.061406   
2   11.500335  14.461336   15.195307  10.841367  ...  0.097037  0.047022   
3    8.569118  16.803516   10.365263  13.606345  ...  0.100879  0.056193   
4   20.612894  11.518078   20.680067   8.899541  ...  0.072352  0.

Looking pretty good! We can set up the train / test split now.

In [10]:
from sklearn.model_selection import train_test_split

# Get unique files per genre to ensure balanced split
unique_files = df['source_file'].unique()

train_files, test_files = train_test_split(
    unique_files, 
    test_size=0.2, 
    random_state=42
)

train_df = df[df['source_file'].isin(train_files)]
test_df = df[df['source_file'].isin(test_files)]

print(f'Train chunks: {len(train_df)}')
print(f'Test chunks: {len(test_df)}')

# Sanity check — no source file should appear in both
assert len(set(train_df['source_file']) & set(test_df['source_file'])) == 0
print('Clean split without leakage.')

Train chunks: 7985
Test chunks: 1996
Clean split without leakage.


In [11]:
train_df.to_csv('data/train_features.csv', index=False)
test_df.to_csv('data/test_features.csv', index=False)

And done. Can come back to iterate on this but should be able to at least start tinkering with classifiers now.